In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

In [2]:
_df = pd.read_csv('data/chembl_valid_10to30atoms.csv')
display(_df)

,smiles
0,NCCNCCNCCN
1,CCNCCCCNCC
2,CCCCCCCCNC
3,NCCOCCOCCN
4,SCCCCCCCCS
...,...
999778,O=c1[nH]c2cc([N+](=O)[O-])c(NN=CC=Cc3ccc([N+](...
999779,O=C(O)c1cc([N+](=O)[O-])cc([N+](=O)[O-])c1-c1c...
999780,O=[N+]([O-])c1cc(S(=O)(=O)c2cc([N+](=O)[O-])c(...
999781,CC(C)(c1cc([N+](=O)[O-])c(O)c([N+](=O)[O-])c1)...


### 1) Get a training set of 20,000 and a holdout set of 2,000 ...

In [3]:
%%time
import random
random.seed(666)

N = 20000

idc = random.sample(range(len(_df)), N+N//10)
train_idc = random.sample(idc, N)
test_idc = [i for i in idc if i not in train_idc]

print(len(train_idc), len(test_idc))
print(set(train_idc).isdisjoint(set(test_idc)))

20000 2000
True
CPU times: user 4.5 s, sys: 0 ns, total: 4.5 s
Wall time: 4.5 s


### 2) Get balanced augmentations for test set and then training set ... 

In [7]:
df_train = _df.iloc[train_idc].reset_index(drop=True)
df_test = _df.iloc[test_idc].reset_index(drop=True)

WHICH = 'train'

In [8]:
from utilities.data_utils import get_anc_to_aug_map
from tqdm import tqdm

if WHICH=='test':
    anc_to_augs = get_anc_to_aug_map(df_test)
elif WHICH=='train':
    anc_to_augs = get_anc_to_aug_map(df_train)
    
df_augs = []
df_ancs = []

for i,(anc_smi,augs) in tqdm(enumerate(anc_to_augs.items()),
                             total=len(anc_to_augs)):
    
    ser = pd.Series({'smiles':anc_smi}) 
    df_ancs.append(ser)
    
    for aug_smi in augs:
        ser = pd.Series({'anc_idx': i,'smiles':aug_smi}) 
        df_augs.append(ser)
    
df_augs = pd.DataFrame(df_augs)
df_anc = pd.DataFrame(df_ancs)

df_augs.to_csv(f'data/model_ready/01/{WHICH}/augmented_smiles.csv',index=False)
df_anc.to_csv(f'data/model_ready/01/{WHICH}/anchor_smiles.csv',index=False)

100%|██████████| 20000/20000 [00:21<00:00, 943.82it/s] 
